# BioMini Phase I Educational Materials

## Notebook 06 — Dataset and Machine Learning
### Dataset에서 Machine-Learning Model과 Predictor로

Notebook 05에서는 biological object를 numerical representation으로 바꾸는
다음 흐름을 만들었습니다.

```text
Protein
   ↓
ProteinAnalyzer
   ↓
FeatureSet
   ↓
Sample
   ↓
Dataset
   ↓
X / y
```

이번 Notebook에서는 실제로 scikit-learn model을 학습하고,
새로운 Protein을 prediction하는 단계까지 진행합니다.

핵심 흐름은 다음과 같습니다.

```text
Dataset
   ↓
X / y
   ↓
StandardScaler
   ↓
Ridge
   ↓
trained model
   ↓
ProteinPredictor
   ↓
prediction
```

이 Notebook은 BioMini Phase I의 **Step 8 — Machine Learning Integration**을
교육용으로 재구성합니다.

---

## Learning Objectives

이 Notebook을 마치면 다음을 할 수 있어야 합니다.

1. `Dataset`을 scikit-learn용 `X`, `y`로 변환한다.
2. feature scaling의 의미를 설명한다.
3. `StandardScaler`와 `Ridge`를 이용해 간단한 regression model을 학습한다.
4. training과 inference 사이에서 feature-column order가 왜 중요한지 설명한다.
5. trained model을 biological object와 연결하는 wrapper의 필요성을 설명한다.
6. `ProteinPredictor`를 구현한다.
7. prediction workflow를 end-to-end로 실행한다.
8. simple baseline model과 Scientific AI의 차이를 이해한다.
9. canonical BioMini `ProteinPredictor`와 교육용 구현을 비교한다.

## 1. Starting Point — Notebook 05의 결과

Machine learning 관점에서 우리가 원하는 것은 다음입니다.

```text
X = input features
y = target
```

예:

```text
length    molecular_weight    hydrophobic_fraction    ...    target
  9             ...                  ...                      0.25
  8             ...                  ...                      0.92
  6             ...                  ...                      0.10
```

여기서 target은 실제 실험값일 수도 있고,
교육용 synthetic value일 수도 있습니다.

이번 Notebook에서는 **ML architecture를 학습하는 것**이 목적이므로
작은 synthetic dataset을 사용합니다.

## 2. 필요한 Domain / Feature Class 준비

Notebook 06은 독립 실행이 가능하도록 필요한 최소 구현을 포함합니다.

In [ ]:
from math import isfinite
from numbers import Real

import pandas as pd


class ValidationError(ValueError):
    pass


class DataError(ValueError):
    pass

In [ ]:
class SequenceMolecule:
    def __init__(self, name: str, sequence: str):
        if not isinstance(name, str) or not name.strip():
            raise ValidationError("name must be a non-empty string")

        if not isinstance(sequence, str):
            raise ValidationError("sequence must be a string")

        self.name = name.strip()
        self.sequence = "".join(sequence.split()).upper()

    def __len__(self) -> int:
        return len(self.sequence)

In [ ]:
class AminoAcid:
    mass_dict = {
        "A": 71.0779, "R": 156.1857, "N": 114.1026, "D": 115.0874,
        "C": 103.1429, "Q": 128.1292, "E": 129.1140, "G": 57.0513,
        "H": 137.1393, "I": 113.1576, "L": 113.1576, "K": 128.1723,
        "M": 131.1960, "F": 147.1738, "P": 97.1152, "S": 87.0773,
        "T": 101.1039, "W": 186.2099, "Y": 163.1732, "V": 99.1310,
    }

    acceptable_codes = set(mass_dict)

    def __init__(self, code: str):
        normalized = code.upper()

        if normalized not in self.acceptable_codes:
            raise ValidationError(f"invalid amino acid: {normalized}")

        self.code = normalized

    def get_mass(self) -> float:
        return self.mass_dict[self.code]

In [ ]:
class Protein(SequenceMolecule):
    valid_codes = AminoAcid.acceptable_codes

    def __init__(self, name: str, sequence: str):
        super().__init__(name, sequence)

        invalid_codes = set(self.sequence) - self.valid_codes

        if invalid_codes:
            raise ValidationError(
                f"invalid protein code(s): {sorted(invalid_codes)}"
            )

        self.amino_acids = [
            AminoAcid(code)
            for code in self.sequence
        ]

In [ ]:
class FeatureSet:
    def __init__(
        self,
        name: str,
        features: dict[str, float]
    ):
        if not isinstance(name, str) or not name.strip():
            raise ValidationError(
                "FeatureSet name must be a non-empty string"
            )

        if not isinstance(features, dict) or not features:
            raise ValidationError(
                "features must be a non-empty dictionary"
            )

        normalized_features = {}

        for key, value in features.items():
            if not isinstance(key, str) or not key.strip():
                raise ValidationError(
                    "feature names must be non-empty strings"
                )

            if isinstance(value, bool) or not isinstance(value, Real):
                raise ValidationError(
                    f"feature {key!r} must be numeric"
                )

            numeric_value = float(value)

            if not isfinite(numeric_value):
                raise ValidationError(
                    f"feature {key!r} must be finite"
                )

            normalized_features[key.strip()] = numeric_value

        self.name = name.strip()
        self.features = normalized_features


class Sample:
    def __init__(
        self,
        feature_set: FeatureSet,
        target=None
    ):
        if not isinstance(feature_set, FeatureSet):
            raise ValidationError(
                "feature_set must be a FeatureSet"
            )

        self.feature_set = feature_set
        self.target = target


class Dataset:
    def __init__(self):
        self.samples: list[Sample] = []

    def add(self, sample: Sample):
        if not isinstance(sample, Sample):
            raise ValidationError(
                "dataset entries must be Sample objects"
            )

        self.samples.append(sample)

    def __len__(self):
        return len(self.samples)

    def to_records(self):
        records = []

        for sample in self.samples:
            row = {
                "name": sample.feature_set.name,
                **sample.feature_set.features,
            }

            if sample.target is not None:
                row["target"] = sample.target

            records.append(row)

        return records

    def to_dataframe(self):
        return pd.DataFrame(
            self.to_records()
        )

    def to_xy(self):
        df = self.to_dataframe()

        if "target" not in df.columns:
            raise DataError(
                "dataset has no target"
            )

        return (
            df.drop(columns=["name", "target"]),
            df["target"]
        )

## 3. ProteinAnalyzer 준비

이번 Notebook에서는 Phase I의 canonical handcrafted feature set을 사용합니다.

```text
length
molecular_weight
hydrophobic_fraction
20 amino-acid fractions
```

In [ ]:
class ProteinAnalyzer:
    hydrophobic_codes = set("AILMFWVY")

    def __init__(self, protein: Protein):
        self.protein = protein

    def length(self) -> int:
        return len(self.protein)

    def molecular_weight(self) -> float:
        if not self.protein.amino_acids:
            return 0.0

        return 18.0153 + sum(
            aa.get_mass()
            for aa in self.protein.amino_acids
        )

    def hydrophobic_fraction(self) -> float:
        sequence = self.protein.sequence

        if not sequence:
            return 0.0

        count = sum(
            1
            for code in sequence
            if code in self.hydrophobic_codes
        )

        return count / len(sequence)

    def amino_acid_fraction_features(self) -> dict[str, float]:
        sequence = self.protein.sequence

        if not sequence:
            return {
                f"aa_{code}_fraction": 0.0
                for code in sorted(AminoAcid.acceptable_codes)
            }

        return {
            f"aa_{code}_fraction":
                sequence.count(code) / len(sequence)

            for code in sorted(
                AminoAcid.acceptable_codes
            )
        }

    def to_features(self) -> FeatureSet:
        features = {
            "length": self.length(),
            "molecular_weight": self.molecular_weight(),
            "hydrophobic_fraction": self.hydrophobic_fraction(),
        }

        features.update(
            self.amino_acid_fraction_features()
        )

        return FeatureSet(
            self.protein.name,
            features
        )

## 4. Small Training Dataset 만들기

아래 target은 교육용 synthetic value입니다.

즉 이 Notebook의 목표는 biology claim을 만드는 것이 아니라,
**BioMini architecture와 ML pipeline의 연결을 학습하는 것**입니다.

In [ ]:
protein_data = [
    ("p1", "ACDEFGHIK", 0.20),
    ("p2", "AILMFWVY", 0.92),
    ("p3", "DEKRQN", 0.08),
    ("p4", "MKWVTFISLLFLFSSAYSR", 0.81),
    ("p5", "GGGGAAAAVVVV", 0.66),
    ("p6", "RRRRKKKK", 0.12),
    ("p7", "VVVVLLLLFFFF", 0.95),
    ("p8", "SSSSTTTTNNNN", 0.18),
    ("p9", "ACACACACAC", 0.33),
    ("p10", "MALWMRLLPLLALLALWG", 0.88),
]

dataset = Dataset()

for name, sequence, target in protein_data:
    protein = Protein(
        name,
        sequence
    )

    features = (
        ProteinAnalyzer(
            protein
        ).to_features()
    )

    dataset.add(
        Sample(
            features,
            target=target
        )
    )

print(
    "dataset size:",
    len(dataset)
)

In [ ]:
df = dataset.to_dataframe()
df

## 5. X와 y 만들기

In [ ]:
X, y = dataset.to_xy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print(
    "number of features:",
    X.shape[1]
)

In [ ]:
X.head()

현재 feature 수는:

```text
3 base features
+
20 amino-acid fraction features
=
23 features
```

입니다.

dataset은 매우 작지만, architecture 학습에는 충분합니다.

## 6. Baseline Model — 왜 Ridge인가?

Phase I에서는 복잡한 model보다 **simple baseline**을 사용합니다.

이유는 다음과 같습니다.

```text
목표:
최고 accuracy
아님

목표:
biological object → feature → model → prediction
architecture 이해
```

`Ridge` regression은 linear model에 L2 regularization을 추가한 간단하고 안정적인 baseline입니다.

## 7. Feature Scaling

`length`, `molecular_weight`, fraction feature는 값의 scale이 매우 다릅니다.

예:

```text
length                 tens
molecular_weight       thousands
fraction               0 ~ 1
```

따라서 `StandardScaler`로 feature scale을 정규화합니다.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    X
)

print(
    X_scaled.shape
)

`StandardScaler`는 각 feature를 대략:

```text
mean = 0
standard deviation = 1
```

이 되도록 변환합니다.

중요한 점은 scaler도 **training data에 fit되는 model component**라는 것입니다.

## 8. Ridge Regression 학습

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(
    alpha=1.0
)

ridge.fit(
    X_scaled,
    y
)

print(
    "training complete"
)

## 9. Training Prediction 확인

In [ ]:
predictions = ridge.predict(
    X_scaled
)

result_df = pd.DataFrame({
    "name": df["name"],
    "target": y,
    "prediction": predictions,
})

result_df

이 결과를 실제 scientific performance로 해석하면 안 됩니다.

이 dataset은 작고 synthetic target을 사용하기 때문입니다.

여기서 중요한 것은:

```text
feature matrix
   ↓
scaling
   ↓
model training
   ↓
prediction
```

이라는 software flow입니다.

## 10. Pipeline으로 묶기

scikit-learn에서는 scaler와 model을 `Pipeline`으로 묶을 수 있습니다.

이 방식이 더 안전합니다.

```text
raw X
 ↓
StandardScaler
 ↓
Ridge
```

In [ ]:
from sklearn.pipeline import Pipeline

model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "ridge",
        Ridge(alpha=1.0)
    ),
])

model.fit(
    X,
    y
)

print(
    "pipeline training complete"
)

In [ ]:
pipeline_predictions = model.predict(
    X
)

pd.DataFrame({
    "name": df["name"],
    "target": y,
    "prediction": pipeline_predictions,
})

## 11. 왜 Pipeline이 더 좋은가?

별도로:

```text
scaler.fit()
scaler.transform()
model.fit()
```

를 관리하면 prediction 시 scaler 적용을 잊을 수 있습니다.

Pipeline을 사용하면:

```python
model.predict(X_new)
```

한 번으로:

```text
scaling
+
prediction
```

이 함께 수행됩니다.

따라서 preprocessing과 model을 하나의 inference unit으로 유지할 수 있습니다.

## 12. 새로운 Protein Prediction

새 Protein을 하나 만들어 prediction하려고 합니다.

In [ ]:
new_protein = Protein(
    "new_protein",
    "AVILMFWY"
)

new_features = (
    ProteinAnalyzer(
        new_protein
    ).to_features()
)

new_features.features

scikit-learn model은 DataFrame을 기대하므로 한 row짜리 DataFrame으로 변환합니다.

In [ ]:
X_new = pd.DataFrame([
    new_features.features
])

X_new

그리고 prediction합니다.

In [ ]:
prediction = model.predict(
    X_new
)[0]

print(
    "prediction:",
    float(prediction)
)

## 13. Feature-Column Order Problem

여기서 매우 중요한 engineering 문제가 있습니다.

training에서는 column order가 다음이었다고 가정합니다.

```text
length
molecular_weight
hydrophobic_fraction
aa_A_fraction
aa_C_fraction
...
```

그런데 inference에서 feature dictionary를 다른 order로 만들면 어떻게 될까요?

일부 ML system에서는 wrong feature order가 silently 잘못된 prediction으로 이어질 수 있습니다.

따라서 training 당시의 feature column을 명시적으로 보존해야 합니다.

In [ ]:
feature_columns = list(
    X.columns
)

feature_columns[:8]

prediction input을 만들 때 반드시 training column order를 사용합니다.

In [ ]:
X_new_safe = pd.DataFrame([
    new_features.features
])[
    feature_columns
]

X_new_safe.head()

이것이 canonical BioMini `ProteinPredictor`가
`feature_columns`를 따로 저장하는 핵심 이유입니다.

## 14. ProteinPredictor 만들기

이제 biological object와 trained model 사이의 wrapper를 만듭니다.

```text
Protein
   ↓
ProteinAnalyzer
   ↓
FeatureSet
   ↓
fixed feature columns
   ↓
trained model
   ↓
float prediction
```

In [ ]:
class ProteinPredictor:
    def __init__(
        self,
        model,
        feature_columns
    ):
        self.model = model
        self.feature_columns = list(
            feature_columns
        )

    def predict(
        self,
        protein: Protein
    ) -> float:
        features = (
            ProteinAnalyzer(
                protein
            )
            .to_features()
            .features
        )

        X = pd.DataFrame([
            features
        ])[
            self.feature_columns
        ]

        return float(
            self.model.predict(X)[0]
        )

In [ ]:
predictor = ProteinPredictor(
    model,
    feature_columns
)

prediction = predictor.predict(
    new_protein
)

print(
    "ProteinPredictor result:",
    prediction
)

## 15. Predictor Wrapper의 의미

이 wrapper 덕분에 사용자는 inference 시 다음을 몰라도 됩니다.

```text
어떤 feature를 계산하는가?
어떤 column order를 사용하는가?
DataFrame을 어떻게 만드는가?
scikit-learn predict를 어떻게 호출하는가?
```

사용자는 다음만 수행합니다.

```python
predictor.predict(protein)
```

즉 `ProteinPredictor`는:

```text
biological domain
↔
machine-learning model
```

사이의 작은 adapter 역할을 합니다.

## 16. 여러 Protein Prediction

In [ ]:
test_proteins = [
    Protein(
        "test_A",
        "AILMFWVY"
    ),
    Protein(
        "test_B",
        "DEKRQN"
    ),
    Protein(
        "test_C",
        "GGGGAAAAVVVV"
    ),
]

for protein in test_proteins:
    value = predictor.predict(
        protein
    )

    print(
        protein.name,
        "->",
        value
    )

## 17. 아주 간단한 Evaluation

교육 목적으로 training error를 확인해 봅니다.

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

train_pred = model.predict(
    X
)

mae = mean_absolute_error(
    y,
    train_pred
)

mse = mean_squared_error(
    y,
    train_pred
)

r2 = r2_score(
    y,
    train_pred
)

print("MAE:", mae)
print("MSE:", mse)
print("R2 :", r2)

주의:

이 수치는 **training dataset 자체에서 계산한 값**입니다.

따라서 이것을 generalization performance로 해석하면 안 됩니다.

실제 ML project에서는:

```text
train / validation / test split
cross-validation
external validation
```

등이 필요합니다.

하지만 Phase I에서는 model evaluation theory보다
software architecture 연결에 집중합니다.

## 18. Train/Test Split 간단 예제

그래도 실제 workflow의 형태를 보기 위해 작은 예제를 실행합니다.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
)

split_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "ridge",
        Ridge(alpha=1.0)
    ),
])

split_model.fit(
    X_train,
    y_train
)

test_pred = split_model.predict(
    X_test
)

print(
    "test MAE:",
    mean_absolute_error(
        y_test,
        test_pred
    )
)

dataset이 매우 작기 때문에 이 값 자체에는 큰 scientific 의미가 없습니다.

이 cell의 목적은 단지 실제 ML workflow에서
training data와 evaluation data를 분리한다는 개념을 보여주는 것입니다.

## 19. Model Complexity보다 Architecture

Phase I에서는 일부러 다음 model을 사용하지 않습니다.

```text
deep neural network
transformer
foundation model
```

이유는 아직 다음 구조를 확실히 만드는 단계이기 때문입니다.

```text
Biological Object
      ↓
Analysis
      ↓
Representation
      ↓
Dataset
      ↓
Model
      ↓
Prediction
```

이 구조가 안정적이면 Phase II에서 model만 더 복잡하게 바꿀 수 있습니다.

## 20. Phase II와의 연결

Phase II에서는 handcrafted feature 대신 다음 representation을 사용할 수 있습니다.

```text
k-mer
one-hot encoding
tokenization
learned embedding
foundation-model embedding
```

하지만 upper-level architecture는 유지할 수 있습니다.

```text
Protein
 ↓
representation
 ↓
model
 ↓
prediction
```

즉 Phase I은 future Scientific AI를 위한 interface foundation입니다.

## 21. Self-Check

In [ ]:
X, y = dataset.to_xy()

assert len(X) == len(y)
assert list(X.columns) == feature_columns

predictor = ProteinPredictor(
    model,
    feature_columns
)

check_protein = Protein(
    "check",
    "AILMFWVY"
)

value = predictor.predict(
    check_protein
)

assert isinstance(
    value,
    float
)

assert len(
    predictor.feature_columns
) == X.shape[1]

print(
    "Notebook 06 self-check: PASS"
)

## 22. Canonical BioMini와 비교

실제 BioMini package에서는 `ProteinPredictor`가 같은 핵심 아이디어를 사용합니다.

canonical implementation의 핵심 구조는:

```python
features = ProteinAnalyzer(protein).to_features().features
X = pd.DataFrame([features])[self.feature_columns]
return float(self.model.predict(X)[0])
```

입니다.

즉 이번 Notebook에서 직접 만든 educational `ProteinPredictor`는
production architecture의 핵심 원리를 그대로 재현합니다.

In [ ]:
try:
    from biomini import (
        Protein as BioMiniProtein,
        ProteinAnalyzer as BioMiniProteinAnalyzer,
        Sample as BioMiniSample,
        Dataset as BioMiniDataset,
        ProteinPredictor as BioMiniProteinPredictor,
    )

    canonical_dataset = BioMiniDataset()

    for name, sequence, target in protein_data:
        p = BioMiniProtein(
            name,
            sequence
        )

        fs = (
            BioMiniProteinAnalyzer(
                p
            ).to_features()
        )

        canonical_dataset.add(
            BioMiniSample(
                fs,
                target=target
            )
        )

    canonical_X, canonical_y = (
        canonical_dataset.to_xy()
    )

    canonical_model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=1.0)
        ),
    ])

    canonical_model.fit(
        canonical_X,
        canonical_y
    )

    canonical_predictor = (
        BioMiniProteinPredictor(
            canonical_model,
            canonical_X.columns
        )
    )

    canonical_test = BioMiniProtein(
        "canonical_test",
        "AILMFWVY"
    )

    print(
        "Canonical prediction:",
        canonical_predictor.predict(
            canonical_test
        )
    )

except ImportError:
    print(
        "BioMini package is not installed in this environment.\n"
        "From the repository root, run:\n"
        "    python -m pip install -e ."
    )

## 23. 현재 단계에서 아직 해결되지 않은 문제

trained model을 만들었습니다.

하지만 Python session을 종료하면 model은 사라집니다.

또 다음 정보도 함께 보존해야 합니다.

```text
trained model
feature columns
target information
model metadata
```

model만 저장하면 충분하지 않습니다.

또 biological object도:

```text
DNA
RNA
Protein
FeatureSet
Sample
Dataset
```

을 저장하고 복원할 필요가 있습니다.

이 문제가 Notebook 07의 출발점입니다.

## 24. What We Learned

이번 Notebook의 핵심 흐름:

```text
Dataset
 ↓
X / y
 ↓
feature scaling
 ↓
Ridge baseline
 ↓
Pipeline
 ↓
feature-column preservation
 ↓
ProteinPredictor
 ↓
biological-object prediction
```

가장 중요한 engineering lesson은 다음입니다.

> trained model만으로는 안정적인 inference system이 완성되지 않는다.

training 당시의 feature schema와 preprocessing context를 함께 보존해야 합니다.

## 25. Bridge to Notebook 07

다음 Notebook에서는:

```text
serialization
schema version
stable identity
metadata
provenance
lineage
ModelArtifact
joblib persistence
```

를 다룹니다.

즉 다음 주제는:

> **Persistence, Provenance and Reproducibility**

입니다.

## 26. Exercises

### Exercise 1 — Baseline Training
현재 dataset으로 `Ridge(alpha=0.1)`, `Ridge(alpha=1.0)`,
`Ridge(alpha=10.0)`을 각각 학습하고 prediction을 비교하십시오.

### Exercise 2 — Feature Scaling
`StandardScaler`를 제거한 model과 결과를 비교하십시오.
왜 차이가 생길 수 있는지 설명하십시오.

### Exercise 3 — New Protein
새 Protein을 하나 만들고 `ProteinPredictor`로 prediction하십시오.

### Exercise 4 — Feature Column Order
`feature_columns`의 순서를 바꿔서 prediction input을 만들면
왜 위험한지 설명하십시오.

### Exercise 5 — Alternative Model
`LinearRegression` 또는 `RandomForestRegressor`를 사용해 같은 Dataset을 학습해 보십시오.

### Exercise 6 — Train/Test Split
`random_state`를 바꾸어 test MAE가 얼마나 달라지는지 확인하십시오.
작은 dataset에서 왜 결과가 불안정한지 설명하십시오.

### Exercise 7 — Architecture Question
왜 `Protein.predict()` method를 Protein class 안에 직접 넣지 않고
별도 `ProteinPredictor`를 사용하는 것이 좋은지 설명하십시오.

### Exercise 8 — Scientific AI Preview
Phase II에서 handcrafted feature를 foundation-model embedding으로 바꾼다면
현재 architecture 중 어떤 부분을 유지하고 어떤 부분을 바꾸어야 할지 생각해 보십시오.